In [1]:
# Blackjack
# Od 1 do 7 graczy współzawodniczy z rozdającym
# nie tworzyłem klasy shoe. Utworzyłem obiekt shoe klasy deck i ona pełni tę funkcję. Kart już nie brakuje. 
# Można by się połasić o obiekt zbierający wykorzystane karty, który by można shuflować i zastąpić nim shoe. Czy to będzie zjadać dużo pamięci? czy na dłuższą metę coś zmieni?
# Jeśli zmieni, to niewiele, bo to będzie 6 talii dalej. Pamieci chyba nie zje dużo. 

import karty, gry     

class BJ_Card(karty.Card):
    """ Karta do blackjacka. """
    ACE_VALUE = 1

    @property
    def value(self):
        if self.is_face_up:
            v = BJ_Card.RANKS.index(self.rank) + 1
            if v > 10:
                v = 10
        else:
            v = None
        return v

class BJ_Deck(karty.Deck):
    """ Talia kart do blackjacka. """
    def populate(self):
        for suit in BJ_Card.SUITS: 
            for rank in BJ_Card.RANKS: 
                self.cards.append(BJ_Card(rank, suit))

        

class BJ_Hand(karty.Hand):
    """ Ręka w blackjacku. """
    def __init__(self, name):
        super(BJ_Hand, self).__init__()
        self.name = name

    def __str__(self):
        rep = self.name + ":\t" + super(BJ_Hand, self).__str__()  
        if self.total:
            rep += "(" + str(self.total) + ")"        
        return rep

    @property     
    def total(self):
        # jeśli karta w ręce ma wartość None, to i wartość sumy wynosi None
        for card in self.cards:
            if not card.value:
                return None
        
        # zsumuj wartości kart, traktuj każdego asa jako 1
        t = 0
        for card in self.cards:
              t += card.value

        # ustal, czy ręka zawiera asa
        contains_ace = False
        for card in self.cards:
            if card.value == BJ_Card.ACE_VALUE:
                contains_ace = True
                
        # jeśli ręka zawiera asa, a suma jest wystarczająco niska,
        # potraktuj asa jako 11
        if contains_ace and t <= 11:
            # dodaj tylko 10, ponieważ już dodaliśmy 1 za asa
            t += 10   
                
        return t

    def is_busted(self):
        return self.total > 21


class BJ_Player(BJ_Hand):
    """ Gracz w blackjacku. """
    def is_hitting(self):
        response = gry.ask_yes_no("\n" + self.name + ", chcesz dobrać kartę? (T/N): ")
        return response == "t"

    def bust(self):
        print(self.name, "ma furę.")
        self.lose()

    def lose(self):
        print(self.name, "przegrywa.")

    def win(self):
        print(self.name, "wygrywa.")

    def push(self):
        print(self.name, "remisuje.")

        
class BJ_Dealer(BJ_Hand):
    """ Rozdający w blackjacku. """
    def is_hitting(self):
        return self.total < 17

    def bust(self):
        print(self.name, "ma furę.")

    def flip_first_card(self):
        first_card = self.cards[0]
        first_card.flip()


class BJ_Game(object):
    """ Gra w blackjacka. """
    def __init__(self, names):      
        self.players = []
        for name in names:
            player = BJ_Player(name)
            self.players.append(player)

        self.dealer = BJ_Dealer("Rozdający")

        self.deck = BJ_Deck()
        self.shoe = BJ_Deck()
        self.deck.populate()
        self.deck.populate()
        self.deck.populate()
        self.deck.shuffle()

    @property
    def still_playing(self):
        sp = []
        for player in self.players:
            if not player.is_busted():
                sp.append(player)
        return sp

    def __additional_cards(self, player):
        while not player.is_busted() and player.is_hitting():
            self.deck.deal([player])
            print(player)
            if player.is_busted():
                player.bust()
           
    def play(self):
        # rozdaj każdemu początkowe dwie karty
        self.deck.deal(self.players + [self.dealer], per_hand = 2)
        self.dealer.flip_first_card()    # ukryj pierwszą kartę rozdającego
        for player in self.players:
            print(player)
        print(self.dealer)

        # rozdaj graczom dodatkowe karty
        for player in self.players:
            self.__additional_cards(player)

        self.dealer.flip_first_card()    # odsłoń pierwszą kartę rozdającego 

        if not self.still_playing:
            # ponieważ wszyscy gracze dostali furę, pokaż tylko rękę rozdającego
            print(self.dealer)
        else:
            # daj dodatkowe karty rozdającemu
            print(self.dealer)
            self.__additional_cards(self.dealer)

            if self.dealer.is_busted():
                # wygrywa każdy, kto jeszcze pozostaje w grze
                for player in self.still_playing:
                    player.win()                    
            else:
                # porównaj punkty każdego gracza pozostającego w grze z punktami rozdającego
                for player in self.still_playing:
                    if player.total > self.dealer.total:
                        player.win()
                    elif player.total < self.dealer.total:
                        player.lose()
                    else:
                        player.push()

        # usuń karty wszystkich graczy
        for player in self.players:
            player.clear()
        self.dealer.clear()

#tu będzie self.deck. sprawdź ilość kart i populate i szuffle. ale shuffle musi być shoe i deck.append(shoe)

        if len(self.deck.cards) < 112:
            self.shoe.populate()
            self.shoe.populate()
            self.shoe.populate()
            self.shoe.shuffle()
            self.deck.cards += self.shoe.cards
            self.shoe.clear()

def main():
    print("\t\tWitaj w grze 'Blackjack'!\n")
    
    names = []
    number = gry.ask_number("Podaj liczbę graczy (1 - 7): ", low = 1, high = 8)
    for i in range(number):
        name = input("Wprowadź nazwę gracza: ")
        names.append(name)
    print()
        
    game = BJ_Game(names)

    again = None
    while again != "n":
        game.play()
        again = gry.ask_yes_no("\nCzy chcesz zagrać ponownie?: ")


main()
input("\n\nAby zakończyć program, naciśnij klawisz Enter.")





		Witaj w grze 'Blackjack'!



Podaj liczbę graczy (1 - 7):  3
Wprowadź nazwę gracza:  Batunia
Wprowadź nazwę gracza:  Fafek
Wprowadź nazwę gracza:  Tata



Batunia:	Ks	6h	(16)
Fafek:	Kh	Kc	(20)
Tata:	5c	9d	(14)
Rozdający:	XX	2c	



Batunia, chcesz dobrać kartę? (T/N):  t


Batunia:	Ks	6h	9d	(25)
Batunia ma furę.
Batunia przegrywa.


...

Tata:	5c	9d	3s	(17)



Tata, chcesz dobrać kartę? (T/N):  t


Tata:	5c	9d	3s	Jh	(27)
Tata ma furę.
Tata przegrywa.
Rozdający:	8d	2c	(10)
Rozdający:	8d	2c	9h	(19)
Fafek wygrywa.



Czy chcesz zagrać ponownie?:  n


Aby zakończyć program, naciśnij klawisz Enter. 


''

In [130]:
import karty, gry     

class BJ_Card(karty.Card):
    """ Karta do blackjacka. """
    ACE_VALUE = 1

    @property
    def value(self):
        if self.is_face_up:
            v = BJ_Card.RANKS.index(self.rank) + 1
            if v > 10:
                v = 10
        else:
            v = None
        return v

class BJ_Deck(karty.Deck):
    """ Talia kart do blackjacka. """
    def populate(self):
        for suit in BJ_Card.SUITS: 
            for rank in BJ_Card.RANKS: 
                self.cards.append(BJ_Card(rank, suit))

# class Shoe(BJ_Deck):
#     def fill_deck(self, deck):
#         deck.cards.append(shoe.cards)
#         shoe.clear()
        

class BJ_Hand(karty.Hand):
    """ Ręka w blackjacku. """
    def __init__(self, name):
        super(BJ_Hand, self).__init__()
        self.name = name

    def __str__(self):
        rep = self.name + ":\t" + super(BJ_Hand, self).__str__()  
        if self.total:
            rep += "(" + str(self.total) + ")"        
        return rep

    @property     
    def total(self):
        # jeśli karta w ręce ma wartość None, to i wartość sumy wynosi None
        for card in self.cards:
            if not card.value:
                return None
        
        # zsumuj wartości kart, traktuj każdego asa jako 1
        t = 0
        for card in self.cards:
              t += card.value

        # ustal, czy ręka zawiera asa
        contains_ace = False
        for card in self.cards:
            if card.value == BJ_Card.ACE_VALUE:
                contains_ace = True
                
        # jeśli ręka zawiera asa, a suma jest wystarczająco niska,
        # potraktuj asa jako 11
        if contains_ace and t <= 11:
            # dodaj tylko 10, ponieważ już dodaliśmy 1 za asa
            t += 10   
                
        return t

    def is_busted(self):
        return self.total > 21


class BJ_Player(BJ_Hand):
    """ Gracz w blackjacku. """
    def is_hitting(self):
        response = gry.ask_yes_no("\n" + self.name + ", chcesz dobrać kartę? (T/N): ")
        return response == "t"

    def bust(self):
        print(self.name, "ma furę.")
        self.lose()

    def lose(self):
        print(self.name, "przegrywa.")

    def win(self):
        print(self.name, "wygrywa.")

    def push(self):
        print(self.name, "remisuje.")

        
class BJ_Dealer(BJ_Hand):
    """ Rozdający w blackjacku. """
    def is_hitting(self):
        return self.total < 17

    def bust(self):
        print(self.name, "ma furę.")

    def flip_first_card(self):
        first_card = self.cards[0]
        first_card.flip()

players = []
for name in names:
    player = BJ_Player(name)
    players.append(player)

dealer = BJ_Dealer("Rozdający")
deck = BJ_Deck()
shoe = BJ_Deck()
deck.populate()
#self.deck.populate()
#self.deck.populate()
deck.shuffle()



In [131]:
print(deck)

2c	4c	4d	10c	Qd	3h	Ad	7s	5s	2h	As	8s	10d	7d	Qc	Jd	7h	8d	3d	6s	Qs	9d	5h	Ks	Qh	Kc	Ah	Kh	4s	3s	9h	Js	Jc	10h	Ac	8c	4h	Kd	Jh	7c	10s	8h	6d	6h	5c	9c	3c	5d	9s	2s	6c	2d	


In [132]:
if len(deck.cards) < 112:
    shoe.populate()
    print("shoe populate\n", shoe)
#self.shoe.populate()
#self.shoe.populate()
    shoe.shuffle()
    print("shoe shuffle\n", shoe)
#    shoe.fill_deck(deck.cards)

shoe populate
 Ac	2c	3c	4c	5c	6c	7c	8c	9c	10c	Jc	Qc	Kc	Ad	2d	3d	4d	5d	6d	7d	8d	9d	10d	Jd	Qd	Kd	Ah	2h	3h	4h	5h	6h	7h	8h	9h	10h	Jh	Qh	Kh	As	2s	3s	4s	5s	6s	7s	8s	9s	10s	Js	Qs	Ks	
shoe shuffle
 3h	7s	8h	6c	5s	Js	3d	5h	6s	4h	9h	9d	Ks	4d	5d	As	4s	4c	10s	Jh	7h	Kh	Qc	3s	5c	Kd	Ac	6d	2c	8d	Ad	10c	Qs	7d	2h	9s	Ah	10d	9c	Qd	8s	Kc	Jc	2s	Jd	7c	10h	3c	6h	2d	Qh	8c	


In [133]:
print(deck)

2c	4c	4d	10c	Qd	3h	Ad	7s	5s	2h	As	8s	10d	7d	Qc	Jd	7h	8d	3d	6s	Qs	9d	5h	Ks	Qh	Kc	Ah	Kh	4s	3s	9h	Js	Jc	10h	Ac	8c	4h	Kd	Jh	7c	10s	8h	6d	6h	5c	9c	3c	5d	9s	2s	6c	2d	


In [134]:
deck.cards += shoe.cards

print(deck)


2c	4c	4d	10c	Qd	3h	Ad	7s	5s	2h	As	8s	10d	7d	Qc	Jd	7h	8d	3d	6s	Qs	9d	5h	Ks	Qh	Kc	Ah	Kh	4s	3s	9h	Js	Jc	10h	Ac	8c	4h	Kd	Jh	7c	10s	8h	6d	6h	5c	9c	3c	5d	9s	2s	6c	2d	3h	7s	8h	6c	5s	Js	3d	5h	6s	4h	9h	9d	Ks	4d	5d	As	4s	4c	10s	Jh	7h	Kh	Qc	3s	5c	Kd	Ac	6d	2c	8d	Ad	10c	Qs	7d	2h	9s	Ah	10d	9c	Qd	8s	Kc	Jc	2s	Jd	7c	10h	3c	6h	2d	Qh	8c	


In [135]:
print(shoe.cards)

print(deck.cards)

[<__main__.BJ_Card object at 0x7fb23c2fcd10>, <__main__.BJ_Card object at 0x7fb23c2fd370>, <__main__.BJ_Card object at 0x7fb23c2fcef0>, <__main__.BJ_Card object at 0x7fb23c2fc470>, <__main__.BJ_Card object at 0x7fb23c2fd2b0>, <__main__.BJ_Card object at 0x7fb23c2fd4f0>, <__main__.BJ_Card object at 0x7fb23c2fc830>, <__main__.BJ_Card object at 0x7fb23c2fcdd0>, <__main__.BJ_Card object at 0x7fb23c2fd310>, <__main__.BJ_Card object at 0x7fb23c2fcd70>, <__main__.BJ_Card object at 0x7fb23c2fcf50>, <__main__.BJ_Card object at 0x7fb23c2fca70>, <__main__.BJ_Card object at 0x7fb23c2fd5b0>, <__main__.BJ_Card object at 0x7fb23c2fc890>, <__main__.BJ_Card object at 0x7fb23c2fc8f0>, <__main__.BJ_Card object at 0x7fb23c2fd130>, <__main__.BJ_Card object at 0x7fb23c2fd250>, <__main__.BJ_Card object at 0x7fb23c2fc3b0>, <__main__.BJ_Card object at 0x7fb23c2fd490>, <__main__.BJ_Card object at 0x7fb23c2fd010>, <__main__.BJ_Card object at 0x7fb23c2fce90>, <__main__.BJ_Card object at 0x7fb23c2fd0d0>, <__main__

In [136]:
shoe.clear()
print(shoe)
print(deck)

<pusta>
2c	4c	4d	10c	Qd	3h	Ad	7s	5s	2h	As	8s	10d	7d	Qc	Jd	7h	8d	3d	6s	Qs	9d	5h	Ks	Qh	Kc	Ah	Kh	4s	3s	9h	Js	Jc	10h	Ac	8c	4h	Kd	Jh	7c	10s	8h	6d	6h	5c	9c	3c	5d	9s	2s	6c	2d	3h	7s	8h	6c	5s	Js	3d	5h	6s	4h	9h	9d	Ks	4d	5d	As	4s	4c	10s	Jh	7h	Kh	Qc	3s	5c	Kd	Ac	6d	2c	8d	Ad	10c	Qs	7d	2h	9s	Ah	10d	9c	Qd	8s	Kc	Jc	2s	Jd	7c	10h	3c	6h	2d	Qh	8c	
